# GeFL Class-Balanced — Full paper-style sweep (Kaggle 2×T4, headless)

**Execution:** Save Version → Save & Run All. Runs headless up to 12h.

**Purpose:** paper-quality sweep for ONE dataset per Kaggle session. Set `DATASET` below and fire.

**Grid per dataset:** 4 mechanisms × 3 imbalance factors × 3 seeds = 36 runs.

| dataset | est. runtime | fits one 12h session? |
|---|---|---|
| mnist / fmnist | ~3-4h | yes |
| cifar10 | ~15h | needs 2 sessions (idempotent rerun) |
| cifar100 | ~18h | needs 2 sessions |

**Idempotent:** completed CSVs are skipped. If a session runs out of 12h clock, rerun the same notebook and it picks up where it left off (add Kaggle output as input dataset in v2 for continuity).

In [ ]:
# ============= SET THIS BEFORE Save & Run All =============
DATASET           = 'mnist'     # 'mnist' | 'fmnist' | 'cifar10' | 'cifar100'
IMBALANCE_FACTORS = [0.01, 0.1, 1.0]
DIR_PARAMS        = [0.3]
SEEDS             = [0, 1, 2]
MECHANISMS        = ['baseline', 'a_only', 'b_only', 'proposed']
GEN_WU_EPOCHS     = 100      # T_KA / 2 (paper)
EPOCHS            = 100      # T_TN   (paper)
# =========================================================

REPO_URL = 'https://github.com/Raunak4518/Fedlearning.git'  # ← your repo
BRANCH   = 'main'

CONFIG_MAP = {
    'mnist':    'configs/mnist_lt.yaml',
    'fmnist':   'configs/fmnist_lt.yaml',
    'cifar10':  'configs/cifar10_lt.yaml',
    'cifar100': 'configs/cifar100_lt.yaml',
}
assert DATASET in CONFIG_MAP, f'DATASET must be one of {list(CONFIG_MAP)}'
CONFIG_PATH = CONFIG_MAP[DATASET]
print(f'DATASET={DATASET}  CONFIG={CONFIG_PATH}')
print(f'Grid: {len(MECHANISMS)} mech × {len(IMBALANCE_FACTORS)} IF × '
      f'{len(DIR_PARAMS)} α × {len(SEEDS)} seeds = '
      f'{len(MECHANISMS)*len(IMBALANCE_FACTORS)*len(DIR_PARAMS)*len(SEEDS)} runs')

In [ ]:
# ----- Clone the code from GitHub -----
import os, subprocess, sys

PROJECT_ROOT = '/kaggle/working/project'
if os.path.exists(PROJECT_ROOT):
    subprocess.check_call(['rm', '-rf', PROJECT_ROOT])

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
    print('Using GITHUB_TOKEN secret (private-repo path).')
except Exception:
    print('No GITHUB_TOKEN secret set — assuming public repo.')

subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', clone_url, PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

commit = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD'], text=True).strip()
print(f'\nCloned {BRANCH} @ {commit[:12]}')

In [ ]:
# ----- Install requirements (skip torch: Kaggle ships GPU-matched build) -----
req = os.path.join(PROJECT_ROOT, 'requirements.txt')
if os.path.exists(req):
    filtered = '/kaggle/working/requirements_no_torch.txt'
    with open(req) as f, open(filtered, 'w') as g:
        for line in f:
            if not line.strip().lower().startswith(('torch', 'torchvision')):
                g.write(line)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', filtered])
print('Deps installed.')

In [ ]:
# ----- GPU sanity: expect 2 T4s -----
import torch
n = torch.cuda.device_count()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| devices:', n)
for i in range(n):
    print(f'  cuda:{i} =', torch.cuda.get_device_name(i))
assert n >= 2, 'Enable Settings → Accelerator → GPU T4 x2.'

In [ ]:
# ----- Build the (config -> cmd) plan, then run pairs in parallel on 2 GPUs -----
import os, sys, subprocess, time, shlex, itertools
import pandas as pd

os.makedirs('./logs', exist_ok=True)
LOG_DIR = f'./logs/{DATASET}'
os.makedirs(LOG_DIR, exist_ok=True)

MECH_FLAGS = {'baseline': (0, 0), 'a_only': (1, 0), 'b_only': (0, 1), 'proposed': (1, 1)}

def _config_cmd(mech, imb, dir_p, seed, out_csv):
    mech_a, mech_b = MECH_FLAGS[mech]
    run_name = f'{DATASET}_mech-{mech}_imb{imb}_dir{dir_p}_seed{seed}'
    return [
        sys.executable, '-m', 'engine_cli_shim',  # placeholder — we invoke GeFL_CVAE.py-equivalent below
    ]

# We don't have a single-run CLI; use scripts/sweep.py once per (mech, imb, seed) so each cmd
# produces exactly one CSV row. sweep.py already accepts space-separated lists of each.
def _run_cmd(mech, imb, dir_p, seed, out_csv):
    return [
        sys.executable, 'scripts/sweep.py',
        '--config', CONFIG_PATH,
        '--imbalance_factors', str(imb),
        '--dir_params', str(dir_p),
        '--seeds', str(seed),
        '--mechanisms', mech,
        '--out_csv', out_csv,
        '--gen_wu_epochs', str(GEN_WU_EPOCHS),
        '--epochs', str(EPOCHS),
        '--sample_test', '5',
        '--eval_centralized_upper_bound', '0',
    ]

# One CSV per single run for maximum idempotence + resumability.
plan = []
for mech, imb, dir_p, seed in itertools.product(MECHANISMS, IMBALANCE_FACTORS, DIR_PARAMS, SEEDS):
    tag = f'{mech}_imb{imb}_dir{dir_p}_seed{seed}'.replace('.', 'p')
    out_csv = f'{LOG_DIR}/{tag}.csv'
    plan.append((tag, out_csv, _run_cmd(mech, imb, dir_p, seed, out_csv)))

todo = [(t, c, cmd) for (t, c, cmd) in plan if not os.path.exists(c)]
print(f'Plan: {len(plan)} total; {len(plan)-len(todo)} already done; {len(todo)} to run.')

# Chunk into pairs (2 concurrent GPUs).
pairs = [todo[i:i+2] for i in range(0, len(todo), 2)]
print(f'{len(pairs)} pairs to fire.')

def _run_pair(pair_idx, entries):
    procs, logs = [], []
    for gpu, (tag, csv_path, cmd) in enumerate(entries):
        env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        log_path = f'/kaggle/working/{DATASET}_pair{pair_idx:03d}_gpu{gpu}.log'
        lf = open(log_path, 'a')
        lf.write(f'\n\n==== {" ".join(map(shlex.quote, cmd))} ====\n'); lf.flush()
        p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
        procs.append(p); logs.append(lf)
        print(f'  cuda:{gpu}  {tag}  pid={p.pid}')
    while any(p.poll() is None for p in procs):
        time.sleep(600)
        state = ', '.join(f'gpu{i}:{"done" if p.poll() is not None else "running"}' for i, p in enumerate(procs))
        print(f'    {time.strftime("%H:%M:%S")}  {state}')
    for lf in logs: lf.close()
    for i, p in enumerate(procs):
        assert p.returncode == 0, f'gpu{i} failed rc={p.returncode} — inspect its log'
    # peek at each
    for (tag, csv_path, _) in entries:
        try:
            d = pd.read_csv(csv_path).iloc[-1]
            print(f'    {tag}  overall={d.get("acc_overall",0):.3f}  head={d.get("acc_head",0):.3f}  '
                  f'tail={d.get("acc_tail",0):.3f}  genLA={d.get("gen_label_accuracy",float("nan"))}')
        except Exception as e:
            print(f'    {tag}  (peek failed: {e})')

for i, entries in enumerate(pairs):
    print(f'\n=== Pair {i+1}/{len(pairs)} ===')
    t0 = time.time()
    _run_pair(i, entries)
    print(f'  pair wall: {(time.time()-t0)/60:.1f} min')

In [ ]:
# ----- Aggregate all per-run CSVs, tolerant of missing columns -----
import glob, pandas as pd, os, re, numpy as np

def _parse_tag(path):
    base = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r'(?P<mech>[a-z_]+?)_imb(?P<imb>[0-9p]+)_dir(?P<dir>[0-9p]+)_seed(?P<seed>[0-9]+)', base)
    if not m: return None
    d = m.groupdict()
    d['imb']  = float(d['imb'].replace('p', '.'))
    d['dir']  = float(d['dir'].replace('p', '.'))
    d['seed'] = int(d['seed'])
    return d

frames = []
for path in sorted(glob.glob(f'{LOG_DIR}/*.csv')):
    tag = _parse_tag(path)
    if tag is None: continue
    try:
        d = pd.read_csv(path)
        for k, v in tag.items(): d[k] = v
        d['dataset'] = DATASET
        frames.append(d)
    except Exception as e:
        print(f'skip {path}: {e}')

if not frames:
    print('NO RESULTS. Sweep did not produce any CSVs.')
else:
    df = pd.concat(frames, ignore_index=True)
    # Tolerant column access
    wanted = ['dataset', 'mech', 'imb', 'dir', 'seed',
              'acc_overall', 'acc_head', 'acc_medium', 'acc_tail',
              'macro_f1', 'class_balanced_accuracy',
              'gen_label_accuracy', 'gen_mean_confidence', 'mnd_ratio']
    cols = [c for c in wanted if c in df.columns]
    print('=== All runs ===')
    print(df[cols].sort_values(['mech', 'imb', 'seed']).to_string(index=False))

    # Mean ± std across seeds per (mech, imb, dir)
    numeric_cols = [c for c in cols if c not in ('dataset', 'mech', 'imb', 'dir', 'seed')]
    agg = df.groupby(['mech', 'imb', 'dir'])[numeric_cols].agg(['mean', 'std']).round(4)
    print('\n=== Mean ± std across seeds ===')
    print(agg.to_string())

    # Save merged table
    out = f'{LOG_DIR}/_merged.csv'
    df.to_csv(out, index=False)
    print(f'\nMerged: {out}')

In [ ]:
# ----- Bundle for download -----
import shutil
bundle_dir = '/kaggle/working/gefl_full_sweep_bundle'
if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir)
if os.path.exists('./logs'):
    shutil.copytree('./logs', os.path.join(bundle_dir, 'logs'))
shutil.make_archive(f'/kaggle/working/gefl_{DATASET}_sweep', 'zip', bundle_dir)
print(f'Bundle: /kaggle/working/gefl_{DATASET}_sweep.zip')